In [12]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import itertools
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             recall_score, f1_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from preprocessing import prepare_data


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# kNN

In [2]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
K_GRID = list(range(3, 52, 2))
WEIGHTS = ["uniform", "distance"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]



def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras.
    A validation fica com os dados mais recentes de train_val."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit
    (aplicado também a *others). Devolve arrays NumPy."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    """Limiar que maximiza o F1 (escolhido na validation)."""
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def run_knn(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)          # scaling só com o train

    best = {"auc": -np.inf}
    for w in WEIGHTS:
        for k in K_GRID:
            m = KNeighborsClassifier(n_neighbors=k, weights=w).fit(Atr, ytr)
            p_val = m.predict_proba(Ava)[:, 1]
            auc = roc_auc_score(yva, p_val)
            if verbose:
                print(f"  w={w:8s} k={k:2d}  val AUC={auc:.4f}")
            if auc > best["auc"]:                       # empate: fica o 1º (k menor)
                best = {"auc": auc, "k": k, "weights": w, "p_val": p_val}

    # limiar de decisão para recall/precision/F1: escolhido na validation
    thr = best_f1_threshold(yva, best["p_val"])

    # modelo final: refit em train+val ou só em train
    if refit:
        Afit, Ate = standardize(Xtv, Xte)         # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = KNeighborsClassifier(n_neighbors=best["k"], weights=best["weights"])
    final.fit(Afit, yfit)
    p_test = final.predict_proba(Ate)[:, list(final.classes_).index(1)]
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "pos_train": int(ytr.sum()), "pos_val": int(yva.sum()),
            "pos_test": int(yte.sum()),
            "k": best["k"], "weights": best["weights"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_knn(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 200)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "k", "weights", "val_AUC", "test_AUC",
            "test_PR_AUC", "thr", "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy  k  weights  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows 13 distance    0.780     0.725        0.392 0.055   0.857      0.316 0.462
    recent     drop_features 19  uniform    0.783     0.820        0.510 0.158   0.857      0.450 0.590
    recent     global_median 47 distance    0.745     0.779        0.535 0.122   0.714      0.357 0.476
    recent     season_median 45 distance    0.744     0.794        0.545 0.107   0.810      0.378 0.515
    recent doy_window_median 11 distance    0.777     0.780        0.536 0.079   0.905      0.404 0.559
stratified         drop_rows 43 distance    0.810     0.803        0.408 0.116   0.889      0.320 0.471
stratified     drop_features 37 distance    0.890     0.795        0.432 0.219   0.444      0.400 0.421
stratified     global_median 33  uniform    0.838     0.803        0.414 0.152   0.852      0.319 0.465
stratified     season_median 27  uniform    0.855     0.785     

# Naive Bayes

In [4]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
VAR_SMOOTHING = [1e-9, 1e-6, 1e-3, 1e-2, 1e-1, 1.0]
ALPHAS = [0.1, 1.0]                                     # suavização de Laplace (Bernoulli)
VARIANTS = ["gaussian", "mixed"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]
BIN_PREFIXES = ("season_", "cluster_")


# ----------------------------------------------------------------------------
# Funções Auxiliares (iguais às do kNN)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


# ----------------------------------------------------------------------------
# Naive Bayes misto: Gaussian (contínuas) + Bernoulli (binárias)
# ----------------------------------------------------------------------------
class MixedNB:
    """P(c|x) ∝ P(c) · Π_contínuas N(x_j|μ_cj,σ²_cj) · Π_binárias Bern(x_j|p_cj).
    O prior aparece nos dois estimadores, por isso subtrai-se uma vez."""

    def __init__(self, cont_idx, bin_idx, var_smoothing=1e-9, alpha=1.0):
        self.cont_idx, self.bin_idx = cont_idx, bin_idx
        self.var_smoothing, self.alpha = var_smoothing, alpha

    def fit(self, A, y):
        self.g_ = GaussianNB(var_smoothing=self.var_smoothing).fit(A[:, self.cont_idx], y)
        self.b_ = BernoulliNB(alpha=self.alpha, binarize=0.5).fit(A[:, self.bin_idx], y)
        self.classes_ = self.g_.classes_
        return self

    def predict_proba(self, A):
        jll = (self.g_.predict_joint_log_proba(A[:, self.cont_idx])
               + self.b_.predict_joint_log_proba(A[:, self.bin_idx])
               - np.log(self.g_.class_prior_))
        jll -= jll.max(axis=1, keepdims=True)           # estabilidade numérica
        p = np.exp(jll)
        return p / p.sum(axis=1, keepdims=True)


def make_model(variant, columns, var_smoothing, alpha):
    if variant == "gaussian":
        return GaussianNB(var_smoothing=var_smoothing)
    cols = list(columns)
    bin_idx = [i for i, c in enumerate(cols) if c.startswith(BIN_PREFIXES)]
    cont_idx = [i for i in range(len(cols)) if i not in bin_idx]
    return MixedNB(cont_idx, bin_idx, var_smoothing, alpha)


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_nb(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    cols = Xtv.columns
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    best = {"auc": -np.inf}
    for variant in VARIANTS:
        alphas = ALPHAS if variant == "mixed" else [None]
        for vs in VAR_SMOOTHING:
            for a in alphas:
                m = make_model(variant, cols, vs, a).fit(Atr, ytr)
                p_val = p1(m, Ava)
                auc = roc_auc_score(yva, p_val)
                if verbose:
                    print(f"  {variant:8s} vs={vs:.0e} a={a}  val AUC={auc:.4f}")
                if auc > best["auc"]:                   # empate: fica o 1º
                    best = {"auc": auc, "variant": variant, "var_smoothing": vs,
                            "alpha": a, "p_val": p_val}

    thr = best_f1_threshold(yva, best["p_val"])

    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = make_model(best["variant"], cols, best["var_smoothing"], best["alpha"])
    final.fit(Afit, yfit)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "variant": best["variant"], "var_smoothing": best["var_smoothing"],
            "alpha": best["alpha"], "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_nb(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 220)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "variant", "var_smoothing", "alpha",
            "val_AUC", "test_AUC", "test_PR_AUC", "thr", "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy  variant  var_smoothing  alpha  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows    mixed          1.000    0.1    0.827     0.801        0.503 0.113   0.952      0.426 0.588
    recent     drop_features    mixed          1.000    1.0    0.812     0.799        0.442 0.083   1.000      0.412 0.583
    recent     global_median    mixed          1.000    1.0    0.810     0.797        0.487 0.090   1.000      0.412 0.583
    recent     season_median    mixed          1.000    1.0    0.807     0.799        0.489 0.094   1.000      0.420 0.592
    recent doy_window_median    mixed          1.000    1.0    0.806     0.799        0.489 0.071   1.000      0.396 0.568
stratified         drop_rows gaussian          0.000    NaN    0.876     0.752        0.341 1.000   0.815      0.289 0.427
stratified     drop_features gaussian          0.001    NaN    0.879     0.747        0.313 0.359   0.889      0.289 0.436
stratified     g

# SVM

In [7]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
C_GRID = [0.01, 0.1, 1, 10, 100]
GAMMA_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]              # só para o kernel rbf
CLASS_WEIGHTS = [None, "balanced"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]


def configs():
    """Grelha de hiperparâmetros (linear primeiro: em empates fica o mais simples)."""
    for cw in CLASS_WEIGHTS:
        for C in C_GRID:
            yield {"kernel": "linear", "C": C, "gamma": None, "class_weight": cw}
    for cw in CLASS_WEIGHTS:
        for C in C_GRID:
            for g in GAMMA_GRID:
                yield {"kernel": "rbf", "C": C, "gamma": g, "class_weight": cw}


def make_svc(cfg, probability=False):
    kw = {"kernel": cfg["kernel"], "C": cfg["C"],
          "class_weight": cfg["class_weight"],
          "probability": probability, "random_state": 0}
    if cfg["gamma"] is not None:
        kw["gamma"] = cfg["gamma"]
    return SVC(**kw)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às do kNN / NB)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_svm(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    # 1) escolha dos hiperparâmetros pelo AUC (decision_function) na validation
    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_svc(cfg).fit(Atr, ytr)
        auc = roc_auc_score(yva, m.decision_function(Ava))
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg}
    cfg = best["cfg"]

    # 2) probabilidades de validation (Platt) -> limiar de F1
    m_val = make_svc(cfg, probability=True).fit(Atr, ytr)
    p_val = p1(m_val, Ava)
    thr = best_f1_threshold(yva, p_val)

    # 3) modelo final: refit em train+val ou só em train
    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = make_svc(cfg, probability=True).fit(Afit, yfit)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "kernel": cfg["kernel"], "C": cfg["C"], "gamma": cfg["gamma"],
            "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "val_AUC_proba": roc_auc_score(yva, p_val),
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0)}


if __name__ == "__main__":
    rows = [run_svm(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "kernel", "C", "gamma", "class_weight",
            "val_AUC", "val_AUC_proba", "test_AUC", "test_PR_AUC", "thr",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

split_test          strategy kernel   C  gamma class_weight  val_AUC  val_AUC_proba  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows    rbf 100  0.003     balanced    0.860          0.860     0.861        0.650 0.121   0.857      0.462 0.600
    recent     drop_features linear 100    NaN     balanced    0.864          0.864     0.879        0.660 0.090   1.000      0.438 0.609
    recent     global_median    rbf 100  0.010     balanced    0.854          0.853     0.827        0.639 0.074   0.905      0.422 0.576
    recent     season_median    rbf 100  0.010     balanced    0.847          0.847     0.827        0.614 0.213   0.619      0.542 0.578
    recent doy_window_median    rbf 100  0.003     balanced    0.861          0.861     0.854        0.658 0.170   0.762      0.485 0.593
stratified         drop_rows linear   1    NaN     balanced    0.890          0.890     0.823        0.398 0.162   0.778      0.339 0.472
stratified     drop_features    rb

# Random Forest

In [11]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
N_ESTIMATORS = 300
SEED = 0
MAX_DEPTH = [3, 5, 8, None]
MIN_LEAF = [1, 3, 5, 10]
MAX_FEATURES = ["sqrt", 0.5]
CLASS_WEIGHTS = [None, "balanced_subsample"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")


def configs():
    """Grelha (modelos mais simples primeiro: em empates fica o mais simples)."""
    for d, l, mf, cw in itertools.product(MAX_DEPTH, MIN_LEAF[::-1],
                                          MAX_FEATURES, CLASS_WEIGHTS):
        yield {"max_depth": d, "min_samples_leaf": l,
               "max_features": mf, "class_weight": cw}


def make_rf(cfg):
    return RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=SEED,
                                  n_jobs=-1, **cfg)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às dos outros métodos)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, X):
    return model.predict_proba(X)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_rf(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)

    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_rf(cfg).fit(Xtr, ytr)
        p_val = p1(m, Xva)
        auc = roc_auc_score(yva, p_val)
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg, "p_val": p_val}
    cfg = best["cfg"]

    thr = best_f1_threshold(yva, best["p_val"])

    Xfit, yfit = (Xtv, ytv) if refit else (Xtr, ytr)
    final = make_rf(cfg).fit(Xfit, yfit)
    p_test = p1(final, Xte)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "max_depth": cfg["max_depth"], "min_leaf": cfg["min_samples_leaf"],
            "max_features": cfg["max_features"], "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0),
            "_importances": pd.Series(final.feature_importances_, index=Xtv.columns)}


if __name__ == "__main__":
    rows = [run_rf(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "max_depth", "min_leaf", "max_features",
            "class_weight", "val_AUC", "test_AUC", "test_PR_AUC", "thr",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

    # importância média das variáveis (estratégias que usam CHL)
    for sp in SPLIT_TESTS:
        imp = pd.concat([r["_importances"] for r in rows
                         if r["split_test"] == sp and r["strategy"] != "drop_features"],
                        axis=1).mean(axis=1).sort_values(ascending=False)
        print(f"\nImportância média ({sp}):")
        print(imp.round(3).head(8).to_string())

split_test          strategy  max_depth  min_leaf max_features       class_weight  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows        5.0         3         sqrt balanced_subsample    0.816     0.798        0.519 0.295   0.810      0.405 0.540
    recent     drop_features        8.0         1         sqrt balanced_subsample    0.816     0.834        0.563 0.268   0.905      0.432 0.585
    recent     global_median        8.0         1         sqrt balanced_subsample    0.806     0.820        0.552 0.161   0.905      0.404 0.559
    recent     season_median        8.0         1         sqrt balanced_subsample    0.795     0.835        0.579 0.112   1.000      0.396 0.568
    recent doy_window_median        8.0         1         sqrt balanced_subsample    0.807     0.829        0.583 0.150   0.952      0.417 0.580
stratified         drop_rows        3.0         5         sqrt                NaN    0.858     0.769        0.316 0.121   0.889   

# Logistic Regression

In [13]:
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15                       # do dataset global
C_GRID = [0.001, 0.01, 0.1, 1, 10, 100]
PENALTIES = ["l2", "l1"]
CLASS_WEIGHTS = [None, "balanced"]
STRATEGIES = ("drop_rows", "drop_features", "global_median",
              "season_median", "doy_window_median")
SPLIT_TESTS = ("recent", "stratified")
CONT_COLS = ["SST", "UI", "YDOY", "sin_doy", "CHL_log", "CHL_uncertainty"]


def configs():
    """Grelha (C pequeno = mais regularização primeiro: em empates fica o mais simples)."""
    for pen in PENALTIES:
        for cw in CLASS_WEIGHTS:
            for C in C_GRID:
                yield {"penalty": pen, "C": C, "class_weight": cw}


def make_lr(cfg):
    solver = "lbfgs" if cfg["penalty"] == "l2" else "liblinear"
    return LogisticRegression(penalty=cfg["penalty"], C=cfg["C"],
                              class_weight=cfg["class_weight"],
                              solver=solver, max_iter=5000, random_state=0)


# ----------------------------------------------------------------------------
# Auxiliares (iguais às dos outros métodos)
# ----------------------------------------------------------------------------
def split_train_val(X, y, dates, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Separa train/val cronologicamente, em datas inteiras."""
    order = np.argsort(dates.to_numpy(), kind="stable")
    X, y, dates = X.iloc[order], y.iloc[order], dates.iloc[order]
    n_val = int(round(len(X) * val_frac / (train_frac + val_frac)))
    cutoff = dates.iloc[len(X) - n_val]
    is_val = (dates >= cutoff).to_numpy()
    return X[~is_val], y[~is_val], X[is_val], y[is_val]


def standardize(X_fit, *others):
    """Standardiza as contínuas com média/desvio (ddof=0) de X_fit."""
    cont = [c for c in CONT_COLS if c in X_fit.columns]
    mu = X_fit[cont].mean()
    sd = X_fit[cont].std(ddof=0).replace(0, 1.0)

    def f(X):
        X = X.copy().astype(float)
        X[cont] = (X[cont] - mu) / sd
        return X.to_numpy()

    return tuple(f(X) for X in (X_fit,) + others)


def best_f1_threshold(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return float(thr[np.argmax(f1)])


def p1(model, A):
    return model.predict_proba(A)[:, list(model.classes_).index(1)]


# ----------------------------------------------------------------------------
def run_lr(strategy, split_test, refit=True, season="one_hot", verbose=False, **kw):
    Xtv, ytv, Xte, yte, dtv, _ = prepare_data(
        strategy, split_test, one_hot_cluster=True, sin_doy=True,
        season=season, scale=False, return_dates=True, **kw)
    Xtr, ytr, Xva, yva = split_train_val(Xtv, ytv, dtv)
    Atr, Ava = standardize(Xtr, Xva)                    # scaling só com o train

    best = {"auc": -np.inf}
    for cfg in configs():
        m = make_lr(cfg).fit(Atr, ytr)
        p_val = p1(m, Ava)
        auc = roc_auc_score(yva, p_val)
        if verbose:
            print(f"  {cfg}  val AUC={auc:.4f}")
        if auc > best["auc"]:                           # empate: fica o 1º
            best = {"auc": auc, "cfg": cfg, "p_val": p_val}
    cfg = best["cfg"]

    thr = best_f1_threshold(yva, best["p_val"])

    if refit:
        Afit, Ate = standardize(Xtv, Xte)               # scaling recalculado em train+val
        yfit = ytv
    else:
        Afit, Ate = standardize(Xtr, Xte)
        yfit = ytr
    final = make_lr(cfg).fit(Afit, yfit)
    p_test = p1(final, Ate)
    y_pred = (p_test >= thr).astype(int)

    return {"strategy": strategy, "split_test": split_test,
            "n_train": len(Xtr), "n_val": len(Xva), "n_test": len(Xte),
            "penalty": cfg["penalty"], "C": cfg["C"],
            "class_weight": cfg["class_weight"],
            "val_AUC": best["auc"],
            "test_AUC": roc_auc_score(yte, p_test),
            "test_PR_AUC": average_precision_score(yte, p_test),
            "thr": thr,
            "recall": recall_score(yte, y_pred, zero_division=0),
            "precision": precision_score(yte, y_pred, zero_division=0),
            "f1": f1_score(yte, y_pred, zero_division=0),
            "_coef": pd.Series(final.coef_[0], index=Xtv.columns)}


if __name__ == "__main__":
    rows = [run_lr(st, sp) for sp in SPLIT_TESTS for st in STRATEGIES]
    pd.set_option("display.width", 250)
    res = pd.DataFrame(rows)
    show = ["split_test", "strategy", "penalty", "C", "class_weight",
            "val_AUC", "test_AUC", "test_PR_AUC", "thr",
            "recall", "precision", "f1"]
    print(res[show].round(3).to_string(index=False))

    # coeficientes médios (estratégias que usam CHL), ordenados por |valor|
    for sp in SPLIT_TESTS:
        coef = pd.concat([r["_coef"] for r in rows
                          if r["split_test"] == sp and r["strategy"] != "drop_features"],
                         axis=1).mean(axis=1)
        print(f"\nCoeficientes médios ({sp}):")
        print(coef.reindex(coef.abs().sort_values(ascending=False).index)
                  .round(2).head(8).to_string())

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_

split_test          strategy penalty   C class_weight  val_AUC  test_AUC  test_PR_AUC   thr  recall  precision    f1
    recent         drop_rows      l2 100     balanced    0.857     0.868        0.616 0.290   1.000      0.457 0.627
    recent     drop_features      l2  10     balanced    0.866     0.867        0.634 0.420   1.000      0.457 0.627
    recent     global_median      l1  10          NaN    0.848     0.868        0.581 0.198   0.619      0.542 0.578
    recent     season_median      l1 100          NaN    0.844     0.870        0.582 0.206   0.619      0.591 0.605
    recent doy_window_median      l1  10          NaN    0.853     0.878        0.591 0.200   0.667      0.583 0.622
stratified         drop_rows      l1   1     balanced    0.904     0.810        0.390 0.389   0.889      0.338 0.490
stratified     drop_features      l2   1     balanced    0.902     0.779        0.378 0.575   0.630      0.315 0.420
stratified     global_median      l2  10     balanced    0.896  

c:\Users\marga\Documents\tecnico\AAut\Projeto-Machine-Learning\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
